# Step 1.1: Aggregating Patient Cancer Registration Records

## Overview
Consolidates distributed cancer registration EMR files. It scans patient data folders, extracts columns containing diagnostics, staging classifications, and patient age, and merges them into a single cohort dataset.

## Inputs and Outputs
- **Input**: EMR directories matching: `../data/step1_dwh_outputs/p1_pts_parse_whole_spilt_2025_/d*`
- **Output**: `../data/step1_dwh_outputs/step1_all_cancer.parquet`



In [ ]:
import pandas as pd
import glob
import os
import re
import sys
sys.path.insert(0, os.path.abspath("../src"))
from fnpred.dtypes import normalize_types, TypeNormalizationError  # fix column types (2026 extract: numerics as strings, dates as date32)

# ==========================================
# 1. Paths and Configuration
# ==========================================
data_dir = "../data" 
SOURCE_EXTRACT = "p1_pts_parse_whole_spilt_2026"   # was p1_pts_parse_whole_spilt_2025(_)
base_input_dir = f"{data_dir}/step1_dwh_outputs/{SOURCE_EXTRACT}"
output_dir = f"{data_dir}/step1_dwh_outputs"

# List to store DataFrames
df_list = []

# ==========================================
# 2. Folder Traversal and Loop Processing
# ==========================================
# Get target directories (d00, d01, ...)
target_dirs = sorted(glob.glob(os.path.join(base_input_dir, "d*")))

print(f"Detected target directories: {len(target_dirs)}")

for d_path in target_dirs:
    # Extract batch ID (e.g., "d00") from directory path
    batch_id = os.path.basename(d_path)
    
    # Check if batch ID matches "d" + digits pattern
    if not re.match(r'^d\d+$', batch_id):
        continue

    # Generate file path pattern
    # Search for files replacing "d00" with batch_id
    search_pattern = os.path.join(d_path, f"df_{batch_id}_tCANCER_REGISTRATION_*.parquet")
    found_files = glob.glob(search_pattern)

    if not found_files:
        print(f"Warning: {batch_id} folder contains no target files. Skipping.")
        continue
    
    # Use the first file if multiple files match the pattern
    file_path = found_files[0]

    try:
        # Read Parquet file
        df = pd.read_parquet(file_path)

        # Select target columns
        df_canc = df.loc[
            :,
            [
                "hs",
                "PT_ID",
                "DIAGNOSIS_CD",
                "DIAGNOSIS_TEXT",
                "UICC_STAGE_BEFORE",
                "MAIN5_STAGE_BEFORE",
                "UICC_STAGE_AFTER",
                "ICD_O_3_CD",
                "ICD_O_3_TEXT",
                "DIAGNOSIS_DATE_1",
                "DIAGNOSIS_DATE_2",
                "AGE",
            ]
        ]


        # Append to the list
        df_list.append(normalize_types(df_canc, name=batch_id))

    except TypeNormalizationError:
        raise   # stop: unconvertible values must be reviewed, not skipped
    except Exception as e:
        print(f"Error: {batch_id} encountered an issue during processing: {e}")

# ==========================================
# 3. Data Aggregation and Output
# ==========================================
if df_list:
    # Vertically concatenate all DataFrames
    df_all = pd.concat(df_list, ignore_index=True)
    
    # Create output directory if it does not exist
    os.makedirs(output_dir, exist_ok=True)

    # Export as Parquet file
    output_filename = "step1_all_cancer.parquet"
    output_path = os.path.join(output_dir, output_filename)
    
    df_all.to_parquet(output_path, index=False)
    
    print("-" * 30)
    print(f"Processing complete: All data combined.")
    print(f"Output file: {output_path}")
    print(f"Total rows: {len(df_all)}")
else:
    print("No target data was found to process.")

## Check: `DIAGNOSIS_DATE_2` vs `FIRST_VISIT_DATE` as the cancer onset date

The source table also has `FIRST_VISIT_DATE`. Before choosing which column to use as the onset date
(step 2 uses `DIAGNOSIS_DATE_2` as `DX_DATE`: it picks the first diagnosis per patient and drops
chemotherapy given before `DX_DATE`), the cells below compare the two columns. Nothing here changes
`step1_all_cancer.parquet`.

1. Raw dtype and value formats of each column
2. Missing values, unparseable values, and implausible dates
3. Row-by-row difference `FIRST_VISIT_DATE - DIAGNOSIS_DATE_2` (days)
4. Position of each date relative to the patient's first anticancer drug (step 1 drug / injection outputs).
   An onset date should normally come on or before the first anticancer drug.

Patient IDs (`PT_ID`) are not displayed; rows are identified by `hs` only.

In [ ]:
# ==========================================
# A. Re-read the date columns as they are in the source (no type normalization)
# ==========================================
DATE_COLS = ["DIAGNOSIS_DATE_1", "DIAGNOSIS_DATE_2", "FIRST_VISIT_DATE"]
KEY_COLS = ["hs", "DIAGNOSIS_CD", "DIAGNOSIS_TEXT"]

raw_list = []
for d_path in target_dirs:
    batch_id = os.path.basename(d_path)
    if not re.match(r"^d\d+$", batch_id):
        continue
    found = glob.glob(os.path.join(d_path, f"df_{batch_id}_tCANCER_REGISTRATION_*.parquet"))
    if not found:
        continue
    df = pd.read_parquet(found[0])
    missing = [c for c in KEY_COLS + DATE_COLS if c not in df.columns]
    if missing:
        print(f"{batch_id}: columns not found {missing}")
    cols = [c for c in KEY_COLS + DATE_COLS if c in df.columns]
    raw_list.append(df[cols].assign(batch=batch_id))

df_dates = pd.concat(raw_list, ignore_index=True)
DATE_COLS = [c for c in DATE_COLS if c in df_dates.columns]
print(f"Rows: {len(df_dates):,}  Patients (hs): {df_dates['hs'].nunique():,}")
print(df_dates[DATE_COLS].dtypes)

# Value formats: digits -> 9, letters -> A (e.g. '99999999' = YYYYMMDD, '9999-99-99')
def value_pattern(s):
    s = s.dropna().astype(str).str.strip()
    return (s.str.replace(r"\d", "9", regex=True)
             .str.replace(r"[A-Za-z]", "A", regex=True)
             .replace("", "<blank>"))

for c in DATE_COLS:
    print(f"\n--- {c}: value formats (top 10) ---")
    print(value_pattern(df_dates[c]).value_counts().head(10).to_string())
    print("examples:", df_dates[c].dropna().astype(str).unique()[:5].tolist())

In [ ]:
# ==========================================
# B. Parse dates and summarize missing / unparseable / implausible values
# ==========================================
import warnings

def parse_date(s: pd.Series) -> pd.Series:
    """date32, datetime, 'YYYYMMDD', 'YYYY-MM-DD', 'YYYY/MM/DD' -> datetime64 (day). Unparseable -> NaT."""
    if pd.api.types.is_datetime64_any_dtype(s):
        out = s
    else:
        txt = s.astype("string").str.strip().mask(lambda x: x == "")
        is8 = txt.str.fullmatch(r"\d{8}").fillna(False)
        out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
        out[is8] = pd.to_datetime(txt[is8], format="%Y%m%d", errors="coerce")
        rest = ~is8 & s.notna()
        # non-string objects (datetime.date) go to to_datetime as they are
        with warnings.catch_warnings():   # mixed formats are parsed element by element
            warnings.simplefilter("ignore", UserWarning)
            out[rest] = pd.to_datetime(s[rest].map(lambda v: v.strip() if isinstance(v, str) else v),
                                       errors="coerce")
    if getattr(out.dt, "tz", None) is not None:
        out = out.dt.tz_convert("Asia/Tokyo").dt.tz_localize(None)
    return out.dt.normalize()

def is_blank(s: pd.Series) -> pd.Series:
    return s.isna() | s.astype("string").str.strip().eq("").fillna(True)

PLAUSIBLE_MIN = pd.Timestamp("1990-01-01")      # adjust if older registrations are expected
PLAUSIBLE_MAX = pd.Timestamp.today().normalize()

parsed = pd.DataFrame({c: parse_date(df_dates[c]) for c in DATE_COLS})
summary = []
for c in DATE_COLS:
    blank = is_blank(df_dates[c])
    p = parsed[c]
    summary.append({
        "column": c,
        "n_rows": len(p),
        "missing(%)": round(100 * blank.mean(), 2),
        "unparseable(n)": int((~blank & p.isna()).sum()),
        "min": p.min(), "median": p.median(), "max": p.max(),
        f"< {PLAUSIBLE_MIN.date()}(n)": int((p < PLAUSIBLE_MIN).sum()),
        "future(n)": int((p > PLAUSIBLE_MAX).sum()),
    })
display(pd.DataFrame(summary).set_index("column"))

for c in DATE_COLS:
    bad = df_dates.loc[~is_blank(df_dates[c]) & parsed[c].isna(), c]
    if len(bad):
        print(f"{c}: unparseable values (top 10)")
        print(bad.astype(str).value_counts().head(10).to_string())

# Missing pattern of the two candidates
miss = pd.crosstab(
    is_blank(df_dates["DIAGNOSIS_DATE_2"]).map({False: "DX2 present", True: "DX2 missing"}),
    is_blank(df_dates["FIRST_VISIT_DATE"]).map({False: "FV present", True: "FV missing"}),
    margins=True)
display(miss)

# Registrations per year (diagnosis year distribution)
display(pd.DataFrame({c: parsed[c].dt.year.astype("Int64").value_counts() for c in DATE_COLS})
          .sort_index().fillna(0).astype(int).rename_axis("year"))

In [ ]:
# ==========================================
# C. Difference FIRST_VISIT_DATE - DIAGNOSIS_DATE_2 (days), row by row
# ==========================================
cmp = df_dates[["hs", "DIAGNOSIS_CD", "DIAGNOSIS_TEXT"]].copy()
cmp["DX1"] = parsed.get("DIAGNOSIS_DATE_1")
cmp["DX2"] = parsed["DIAGNOSIS_DATE_2"]
cmp["FV"] = parsed["FIRST_VISIT_DATE"]
cmp["diff_FV_DX2"] = (cmp["FV"] - cmp["DX2"]).dt.days

both = cmp.dropna(subset=["DX2", "FV"])
print(f"Rows with both dates: {len(both):,} / {len(cmp):,}")
print(f"  FV == DX2 : {(both['diff_FV_DX2'] == 0).mean():.1%}")
print(f"  FV <  DX2 : {(both['diff_FV_DX2'] < 0).mean():.1%}  (first visit before diagnosis)")
print(f"  FV >  DX2 : {(both['diff_FV_DX2'] > 0).mean():.1%}  (first visit after diagnosis, e.g. diagnosed elsewhere)")

print("\nQuantiles of FV - DX2 (days):")
print(both["diff_FV_DX2"].quantile([0, .01, .05, .1, .25, .5, .75, .9, .95, .99, 1]).to_string())

bins = [-float("inf"), -366, -91, -31, -1, 0, 30, 90, 365, float("inf")]
labels = ["<= -366", "-365..-91", "-90..-31", "-30..-1", "0", "1..30", "31..90", "91..365", ">= 366"]
diff_bin = pd.cut(both["diff_FV_DX2"], bins=bins, labels=labels)
display(diff_bin.value_counts().reindex(labels)
        .to_frame("n").assign(pct=lambda x: (100 * x["n"] / x["n"].sum()).round(1)))

# Is the difference different by cancer site (ICD-10 3 characters)?
both = both.assign(DX3=both["DIAGNOSIS_CD"].astype(str).str.slice(0, 3))
by_site = (both.groupby("DX3")["diff_FV_DX2"]
               .agg(n="size", **{"equal(%)": lambda x: round(100 * (x == 0).mean(), 1)},
                    median="median", **{"|diff|>90d(%)": lambda x: round(100 * (x.abs() > 90).mean(), 1)})
               .sort_values("n", ascending=False).head(20))
display(by_site)

# Same check against DIAGNOSIS_DATE_1 (for reference)
if cmp["DX1"].notna().any():
    d1 = cmp.dropna(subset=["DX1", "DX2"])
    print(f"\nDIAGNOSIS_DATE_1 == DIAGNOSIS_DATE_2: {(d1['DX1'] == d1['DX2']).mean():.1%} (n={len(d1):,})")
    d1fv = cmp.dropna(subset=["DX1", "FV"])
    print(f"DIAGNOSIS_DATE_1 == FIRST_VISIT_DATE: {(d1fv['DX1'] == d1fv['FV']).mean():.1%} (n={len(d1fv):,})")

# Largest disagreements (hs only, no PT_ID)
print("\nLargest |FV - DX2| (top 20):")
display(both.reindex(both["diff_FV_DX2"].abs().sort_values(ascending=False).index)
            [["hs", "DIAGNOSIS_CD", "DIAGNOSIS_TEXT", "DX1", "DX2", "FV", "diff_FV_DX2"]].head(20))

In [ ]:
# ==========================================
# D. Each candidate date vs the patient's first anticancer drug
#    (uses step1_all_drug_taken.parquet / step1_all_injection.parquet if they exist)
# ==========================================
chemo_files = [os.path.join(output_dir, f) for f in ("step1_all_drug_taken.parquet", "step1_all_injection.parquet")]
chemo_files = [f for f in chemo_files if os.path.exists(f)]

if not chemo_files:
    print("Step 1 drug / injection outputs not found. Run those notebooks first to use this check.")
else:
    chemo = pd.concat([pd.read_parquet(f, columns=["hs", "QUERY_DATE", "ANTI_CANCER_FLG"]) for f in chemo_files],
                      ignore_index=True)
    chemo = chemo[chemo["ANTI_CANCER_FLG"] == "＋"]
    first_chemo = (parse_date(chemo["QUERY_DATE"]).groupby(chemo["hs"]).min().rename("FIRST_CHEMO"))
    print(f"Patients with an anticancer drug: {len(first_chemo):,}")

    # Patient level: earliest date of each candidate across all registrations (as step 2 does with DX_DATE)
    per_pt = cmp.groupby("hs")[["DX2", "FV"]].min().join(first_chemo, how="inner")
    rows = []
    for c, label in [("DX2", "DIAGNOSIS_DATE_2"), ("FV", "FIRST_VISIT_DATE")]:
        gap = (per_pt["FIRST_CHEMO"] - per_pt[c]).dt.days     # >0: date is before first chemo
        g = gap.dropna()
        rows.append({
            "candidate": label,
            "patients": len(per_pt),
            "date missing(%)": round(100 * gap.isna().mean(), 1),
            "date <= first chemo(%)": round(100 * (g >= 0).mean(), 1),
            "date > first chemo(%)": round(100 * (g < 0).mean(), 1),
            "date > first chemo by >30d(%)": round(100 * (g < -30).mean(), 1),
            "gap median(d)": g.median(),
            "gap IQR(d)": f"{g.quantile(.25):.0f} to {g.quantile(.75):.0f}",
            "gap > 365d(%)": round(100 * (g > 365).mean(), 1),
        })
    display(pd.DataFrame(rows).set_index("candidate").T)

    # Which candidate is closer to (and not after) the first anticancer drug?
    pp = per_pt.dropna()
    gap_dx2 = (pp["FIRST_CHEMO"] - pp["DX2"]).dt.days
    gap_fv = (pp["FIRST_CHEMO"] - pp["FV"]).dt.days
    print(f"\nPatients with both candidates and chemo: {len(pp):,}")
    print(f"  DX2 after first chemo, FV not : {((gap_dx2 < 0) & (gap_fv >= 0)).sum():,}")
    print(f"  FV after first chemo, DX2 not : {((gap_fv < 0) & (gap_dx2 >= 0)).sum():,}")
    print(f"  both after first chemo        : {((gap_fv < 0) & (gap_dx2 < 0)).sum():,}")